# Native CLM Architecture Lab

This is the single long-lived notebook for Native CLM architecture search.

Primary objective:

\[
\min_A \; \mathrm{NLL}_{\mathrm{NTP}}(A)
\]

under controlled parameter, data, token, and compute budgets.

The first milestone compares a modern ~2M decoder-only Transformer (`T1`) against a ~2M native CLM (`C0`) and then adds one CLM architectural change at a time.

**Important:** continual-learning mechanisms (growth, mitosis, rollback, online writes) are intentionally excluded from the first phase.


## Protocol

For a direct comparison keep constant:

- tokenizer and vocabulary
- dataset revision/splits
- context length
- train token budget
- optimizer and LR schedule
- seed set

Always report params, PPL, training/inference compute estimates, throughput, and peak memory.

Use development runs to iterate. Freeze a promoted candidate before confirmation seeds. Export promoted records to `results/NCLM-XXX/`.


In [ ]:
from dataclasses import dataclass, asdict
from pathlib import Path
import json, math, os, platform, random, time

try:
    import numpy as np
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except Exception as e:
    raise RuntimeError("Native CLM lab requires PyTorch and NumPy") from e

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "torch:", torch.__version__)


In [ ]:
@dataclass(frozen=True)
class LabConfig:
    seed: int = 91001
    vocab_size: int = 2048
    context_length: int = 128
    d_model: int = 192
    transformer_layers: int = 4
    n_heads: int = 6
    n_kv_heads: int = 2
    ffn_mult: float = 2.5

    # Initial CLM scaffold. These are search variables, not validated optima.
    clm_steps: tuple = (4, 4, 4)
    clm_windows: tuple = (8, 32, 128)

CFG = LabConfig()
print(asdict(CFG))


In [ ]:
def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(CFG.seed)

def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

def nll_to_ppl(nll):
    return float(math.exp(float(nll)))


## T1 — modern small Transformer baseline

This baseline is deliberately modern in structure rather than a historical GPT-Neo-like baseline:

- pre-norm RMSNorm
- RoPE
- SwiGLU
- grouped-query attention
- causal decoder
- tied embedding/LM head

The dimensions are small because this notebook is for fast architecture search. Parameter matching should be checked after every CLM change rather than assumed.


In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps
    def forward(self, x):
        scale = x.pow(2).mean(-1, keepdim=True).add(self.eps).rsqrt()
        return x * scale * self.weight

def apply_rope(x):
    # x: [B,H,T,D], D must be even
    B,H,T,D = x.shape
    half = D // 2
    inv = 1.0 / (10000 ** (torch.arange(0, half, device=x.device, dtype=x.dtype) / half))
    pos = torch.arange(T, device=x.device, dtype=x.dtype)
    ang = torch.einsum("t,d->td", pos, inv)
    sin, cos = ang.sin()[None,None,:,:], ang.cos()[None,None,:,:]
    x1, x2 = x[..., :half], x[..., half:half*2]
    return torch.cat([x1*cos - x2*sin, x1*sin + x2*cos], dim=-1)

class GQAAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.head_dim = d_model // n_heads
        self.q = nn.Linear(d_model, n_heads*self.head_dim, bias=False)
        self.k = nn.Linear(d_model, n_kv_heads*self.head_dim, bias=False)
        self.v = nn.Linear(d_model, n_kv_heads*self.head_dim, bias=False)
        self.o = nn.Linear(n_heads*self.head_dim, d_model, bias=False)
    def forward(self, x):
        B,T,C = x.shape
        q = self.q(x).view(B,T,self.n_heads,self.head_dim).transpose(1,2)
        k = self.k(x).view(B,T,self.n_kv_heads,self.head_dim).transpose(1,2)
        v = self.v(x).view(B,T,self.n_kv_heads,self.head_dim).transpose(1,2)
        q, k = apply_rope(q), apply_rope(k)
        rep = self.n_heads // self.n_kv_heads
        k = k.repeat_interleave(rep, dim=1)
        v = v.repeat_interleave(rep, dim=1)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.o(y.transpose(1,2).contiguous().view(B,T,-1))

class SwiGLU(nn.Module):
    def __init__(self, d_model, hidden):
        super().__init__()
        self.gate = nn.Linear(d_model, hidden, bias=False)
        self.up = nn.Linear(d_model, hidden, bias=False)
        self.down = nn.Linear(hidden, d_model, bias=False)
    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        hidden = int(cfg.d_model * cfg.ffn_mult)
        self.n1 = RMSNorm(cfg.d_model)
        self.attn = GQAAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads)
        self.n2 = RMSNorm(cfg.d_model)
        self.ff = SwiGLU(cfg.d_model, hidden)
    def forward(self, x):
        x = x + self.attn(self.n1(x))
        x = x + self.ff(self.n2(x))
        return x

class ModernTransformerLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.transformer_layers)])
        self.norm = RMSNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok.weight
    def forward(self, ids):
        x = self.tok(ids)
        for b in self.blocks:
            x = b(x)
        return self.lm_head(self.norm(x))

T1 = ModernTransformerLM(CFG).to(DEVICE)
print("T1 params:", count_parameters(T1))


## C0 — Native CLM scaffold

`C0` is a controlled starting point, **not** a claim that this is the historical best CLM.

It uses:

- shared Cell parameters across recurrent steps
- explicit recurrent-step embeddings
- causal local receptive fields
- gated state update
- staged windows `(8, 32, 128)`

This gives the notebook a concrete baseline that can be ablated. The first formal task is to reproduce/align the exact historical CLM baseline from repository artifacts before treating `C0` as canonical.


In [ ]:
def causal_window_mean(x, window):
    # x [B,T,C]; causal moving average including current position
    B,T,C = x.shape
    cs = torch.cat([torch.zeros(B,1,C,device=x.device,dtype=x.dtype), x.cumsum(dim=1)], dim=1)
    idx = torch.arange(T, device=x.device)
    start = (idx - window + 1).clamp_min(0)
    sums = cs[:, idx+1] - cs[:, start]
    denom = (idx - start + 1).to(x.dtype)[None,:,None]
    return sums / denom

class SharedCell(nn.Module):
    def __init__(self, d_model, max_steps):
        super().__init__()
        self.norm = RMSNorm(d_model)
        self.step = nn.Embedding(max_steps, d_model)
        self.cand = SwiGLU(d_model*2, d_model*2)
        self.proj = nn.Linear(d_model*2, d_model, bias=False)
        self.gate = nn.Linear(d_model*2, d_model, bias=True)
        nn.init.constant_(self.gate.bias, 2.0)  # retain-state-favoring initialization; ablate explicitly
    def forward(self, h, msg, step_idx):
        s = self.step.weight[step_idx][None,None,:].expand_as(h)
        z = torch.cat([self.norm(h) + s, msg], dim=-1)
        candidate = torch.tanh(self.proj(self.cand(z)))
        keep = torch.sigmoid(self.gate(z))
        return keep*h + (1.0-keep)*candidate

class NativeCLMLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.max_steps = sum(cfg.clm_steps)
        self.cell = SharedCell(cfg.d_model, self.max_steps)
        self.norm = RMSNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok.weight
    def forward(self, ids):
        h = self.tok(ids)
        step_idx = 0
        for window, n_steps in zip(self.cfg.clm_windows, self.cfg.clm_steps):
            for _ in range(n_steps):
                msg = causal_window_mean(h, window)
                h = self.cell(h, msg, step_idx)
                step_idx += 1
        return self.lm_head(self.norm(h))

C0 = NativeCLMLM(CFG).to(DEVICE)
print("C0 params:", count_parameters(C0))
print("param ratio C0/T1:", count_parameters(C0)[0] / count_parameters(T1)[0])


## Smoke test

This only verifies tensor plumbing and causal next-token loss. It is **not** a scientific benchmark.

Replace the synthetic token batch with the frozen dataset/tokenizer adapter before any promoted result.


In [ ]:
B = 2
x = torch.randint(0, CFG.vocab_size, (B, CFG.context_length), device=DEVICE)
with torch.no_grad():
    t_logits = T1(x[:, :-1])
    c_logits = C0(x[:, :-1])
    y = x[:, 1:]
    t_loss = F.cross_entropy(t_logits.reshape(-1, CFG.vocab_size), y.reshape(-1))
    c_loss = F.cross_entropy(c_logits.reshape(-1, CFG.vocab_size), y.reshape(-1))
print("smoke", {"T1_loss": float(t_loss), "C0_loss": float(c_loss)})


## Experiment registry

Add candidates here instead of creating new notebooks. A candidate is only marked `validated` after controlled multi-seed evidence.

Suggested sequence:

- `C1`: gated-update ablation/optimization
- `C2`: phase-conditioned recurrence
- `C3`: small phase-specific modulation
- `C4`: receptive-field schedule search
- later: dual-state, adaptive recurrence, communication/compute separation, functional specialization


In [ ]:
EXPERIMENTS = [
    {"id": "T1", "family": "transformer", "status": "baseline", "change": "modern small decoder"},
    {"id": "C0", "family": "native-clm", "status": "baseline", "change": "shared recurrent Cell scaffold"},
    {"id": "C1", "family": "native-clm", "status": "planned", "change": "gated-update study"},
    {"id": "C2", "family": "native-clm", "status": "planned", "change": "phase-conditioned recurrence"},
    {"id": "C3", "family": "native-clm", "status": "planned", "change": "phase-specific modulation"},
    {"id": "C4", "family": "native-clm", "status": "planned", "change": "receptive-field schedule"},
]
EXPERIMENTS


## Result row and promotion helper

Promote only controlled runs. Never overwrite a previously promoted `NCLM-XXX` directory.


In [ ]:
def result_row(model_id, model, validation_nll, train_tokens, seed, **extra):
    total, trainable = count_parameters(model)
    row = {
        "model_id": model_id,
        "validation_nll": float(validation_nll),
        "validation_ppl": nll_to_ppl(validation_nll),
        "parameter_count_total": total,
        "parameter_count_trainable": trainable,
        "train_tokens": int(train_tokens),
        "context_length": CFG.context_length,
        "seed": int(seed),
    }
    row.update(extra)
    return row

def promote_run(run_id, config, metrics, root=Path("research/native-clm/results")):
    out = root / run_id
    if out.exists():
        raise FileExistsError(f"immutable promoted run already exists: {out}")
    out.mkdir(parents=True)
    (out / "config.json").write_text(json.dumps(config, indent=2, sort_keys=True))
    (out / "metrics.json").write_text(json.dumps(metrics, indent=2, sort_keys=True))
    env = {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "device": DEVICE,
    }
    (out / "environment.json").write_text(json.dumps(env, indent=2, sort_keys=True))
    (out / "README.md").write_text(
        f"# {run_id}\n\nPromoted from `native_clm_lab.ipynb`. "
        "Do not rewrite config/metrics after promotion.\n"
    )
    return out


## Next execution milestone

Before `NCLM-001` is promoted:

1. wire the frozen TinyStories (or chosen corpus) revision and tokenizer into this notebook;
2. reproduce the historical Native CLM baseline exactly enough to define `C0`;
3. tune `T1` dimensions to achieve a close parameter match instead of relying on the provisional configuration;
4. freeze train-token and optimizer schedules;
5. run development seeds;
6. promote the frozen baseline protocol;
7. only then begin `C1`–`C4`.

The notebook should remain the single architecture workbench throughout those revisions.
